In [78]:
import os 
import re
import json
import logging
import pandas as pd 
import numpy as np
from typing import List, Dict, Any, Optional
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.evaluation import load_evaluator
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_community.vectorstores import Chroma
from backend.app.config import (
    OPENAI_KEY,
    MINI_LM_EMBED,
    GPT_4o,
    OPENAI_EMBED,
    VECTOR_DB_PATH,
    PROJECT_ROOT
)
DEFAULT_EMBED_MODEL = OPENAI_EMBED

In [90]:
question_answer_pairs_german = [
  {
    "question": "Welche weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?",
    "answer_reference": "Empfehlung: MRT bei persistierenden Beschwerden mit unauffälligem Röntgenbefund."
  },
  {
    "question": "Wann sollte eine MRT beider Hüftgelenke durchgeführt werden?",
    "answer_reference": "Empfehlung: MRT beider Hüften bei einseitiger Femurkopfnekrose im ARCO Stadium I-IV."
  },
  {
    "question": "Welche Klassifikation wird zur Stadieneinteilung der atraumatischen Femurkopfnekrose empfohlen?",
    "answer_reference": "Empfehlung: Nutzung der modifizierten ARCO-Klassifikation."
  },
  {
    "question": "Wie sollte bei Verdacht auf eine subchondrale Fraktur im ARCO Stadium II, aber unklarer Diagnostik, weiter vorgegangen werden?",
    "answer_reference": "Empfehlung: Durchführung einer CT zur Klärung der subchondralen Fraktur."
  },
  {
    "question": "Sollte eine Szintigraphie zur Diagnostik der atraumatischen Femurkopfnekrose eingesetzt werden?",
    "answer_reference": "Empfehlung: Szintigraphie wird nicht zur Diagnostik der atraumatischen Femurkopfnekrose empfohlen."
  },
  {
    "question": "Wie differenziert man im MRT zwischen einem transitorischen Knochenmarködem und einer Osteonekrose?",
    "answer_reference": "Empfehlung: MRT-Muster und klinischer Verlauf sind entscheidend für die Differenzierung."
  },
  {
    "question": "Welche bildgebende Methode gilt als Goldstandard in der Diagnostik der atraumatischen Femurkopfnekrose?",
    "answer_reference": "Empfehlung: MRT als Goldstandard."
  },
  {
    "question": "Welche Bildgebung eignet sich am besten zur Detektion einer subchondralen Fraktur?",
    "answer_reference": "Empfehlung: CT für die Darstellung subchondraler Frakturen."
  },
  {
    "question": "Welche Risikofaktoren sprechen für eine bilaterale Beteiligung bei Femurkopfnekrose?",
    "answer_reference": "Empfehlung: Einseitige Femurkopfnekrose erhöht das Risiko für bilaterale Erkrankung; Risikofaktoren beachten."
  },
  {
    "question": "Welche röntgenologischen Befunde charakterisieren das ARCO Stadium III?",
    "answer_reference": "Empfehlung: Zeichen der subchondralen Fraktur mit beginnender Gelenkflächeninkongruenz im Röntgenbild."
  }
]

### Document Extraction & Splitting

In [91]:
def extract_text_from_pdf(file_path: str) -> str:
        """
        Extract text from a PDF file using PyPDFLoader.
        """
        try:
            loader = PyPDFLoader(file_path)
            documents = loader.load()
            return [page.page_content for page in documents]
        except Exception as e: 
            raise Exception(f"Error extracting text from PDF: {e}")


In [ ]:
pdf_path = os.path.join(os.getcwd(), "backend", "app", "documents", "Guideline_atraumatische_Femurkopfnekrose_2019-09_1-abgelaufen.pdf")

pdf_extraction = extract_text_from_pdf(pdf_path)
print(pdf_extraction)

if isinstance(pdf_extraction, list):
    pdf_extraction = "\n".join(pdf_extraction)

#print(pdf_extraction)


### Text Splitting

In [20]:
def split_text(
        pdf_docs: List[str], 
        chunk_size: int = 1000, 
        chunk_overlap: int = 200) -> List[Document]:
    """
    Splits text into chunks for efficient embedding.
    
    - `chunk_size`: Max characters per chunk.
    - `chunk_overlap`: Overlap between chunks for better context retention.
    
    Returns a list of text chunks.
    """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len
    )

    chunks = []
    for page_text in pdf_docs:
        split_chunks = text_splitter.split_text(page_text)
        for chunk in split_chunks:
            chunks.append(Document(page_content=chunk))
    
    return chunks



In [30]:
text_chunks = split_text([cleaned_text], chunk_size=500, chunk_overlap=100)
print(f"Number of chunks: {len(text_chunks)}")

Number of chunks: 945


In [33]:
print(text_chunks[0].page_content)

Langfassung  S3-Leitlinie Atraumatische Femurkopfnekrose des Erwachsenen  Version 2.4 August 2019  AWMF-Register-Nr. 033/050 Atraumatische Femurkopfnekrose des Erwachsenen  Federführender Autor  Prof. Dr. A


### Word Embeddings & Vector Storage

In [ ]:
def store_embeddings(
        docs: List[Document],
        embed_model: Optional[str] = None,
        collection_name: str = "chromadb"
    ) -> None:

    """
    Generates embeddings for text chunks and stores them in ChromaDB.

    Args:
        docs (List[Document]): List of documents to embed.
        embed_model (Optional[str]): Name of the embedding model.  Defaults to OPENAI_EMBED.
        collection_name (str): Name of the ChromaDB collection.  Defaults to "chromadb".
    """

    if embed_model is None:
        embed_model = OPENAI_EMBED

    logging.info(f"Using embedding model: {embed_model}") 

    if embed_model == OPENAI_EMBED:
        embedding_model = OpenAIEmbeddings(model=OPENAI_EMBED, api_key=OPENAI_KEY)
    elif embed_model == MINI_LM_EMBED:
        embedding_model = HuggingFaceEmbeddings(model_name=MINI_LM_EMBED)
    else:
        raise ValueError(f"Unsupported embedding model: {embed_model}") 
    
    text_chunks = [doc.page_content for doc in docs]
    metadata = [doc.metadata for doc in docs]
    logging.info(f"Number of chunks to embed: {len(text_chunks)}") 

    dir_path = os.path.join(VECTOR_DB_PATH, collection_name)
    os.makedirs(dir_path, exist_ok=True)
    logging.info(f"ChromaDB directory: {dir_path}")

    db = Chroma(
        collection_name=collection_name,
        embedding_function=embedding_model, 
        persist_directory=dir_path
    )
    db.add_texts(texts=text_chunks, metadatas=metadata, embeddings=embedding_model) 

    logging.info(f"✅ Stored {len(text_chunks)} text chunks in ChromaDB (Collection: {collection_name})")

In [40]:
store_embeddings(text_chunks, embed_model=OPENAI_EMBED, collection_name="medical_dataset_test")

/var/folders/7t/g4h6fbw915v56stlfs263jzc0000gn/T/ipykernel_44661/4253828258.py:36: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(


In [42]:
def retrieve_text(
        vectordb_path: str,
        query: str,
        embed_model: Optional[str] = None,
        collection_name: str = "chromadb",
        results_to_return: int = 3):
    """
    Retrieve text from the vector store based on the query.

    Args:
        vector_db (Chroma): ChromaDB instance.
        query (str): Query string to search for in the vector store.

    Returns:
        List[Document]: Retrieved documents from the vector store.
    """
    if not os.path.exists(vectordb_path):
        raise FileNotFoundError(f"Vector store not found at: {vectordb_path}")

    if not query:
        raise ValueError("Query cannot be empty.")

    if embed_model is None:
        embed_model = DEFAULT_EMBED_MODEL

    embedding_model = OpenAIEmbeddings(model=embed_model, api_key=OPENAI_KEY)

    print(f"Retrieving from Vector DB Path: {vectordb_path}")
    vector_db = Chroma(
        collection_name=collection_name,
        embedding_function=embedding_model,
        persist_directory=vectordb_path
    )

    matched_texts = vector_db.similarity_search_with_score(query, k=results_to_return)

    print(f"Number of Retrieved Texts: {len(matched_texts)}")
    print(f"Query used for retrieval: {query}")

    return matched_texts

In [46]:
vectordb_path = os.path.join(VECTOR_DB_PATH, "medical_dataset_test")
question = "weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?"
answer_reference = ["MRT bei persistierenden Beschwerden mit unauffälligem Röntgenbefund"] 

matched_texts = retrieve_text(
        vectordb_path=vectordb_path,
        query=question,
        embed_model=OPENAI_EMBED,
        collection_name="medical_dataset_test",
        results_to_return=3
    )

for text, score in matched_texts:
    print(f"Score: {score}", end="\n")
    print(f"Text: {text.page_content}", end="\n")
    print(f"Metadata: {text.metadata}", end="\n")



Retrieving from Vector DB Path: backend/app/vector_store/medical_dataset_test
Number of Retrieved Texts: 3
Query used for retrieval: weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?
Score: 0.578153669834137
Text: . Hinken und Bewegungsschmerz bzw.  Bewegungseinschränkungen, kein Hinweis auf Di fferentialdiagnosen) soll  zunächst eine Röntgenuntersuchung (Beckenübersicht und betroffene Hüfte in  Lauensteinprojektion) durchgeführt werden.  Expertenkonsens (stark) Entscheidung basiert aus-schließlich auf Experten-konsens, 100  Zustimmung  Bei unauffälligem Röntgenbild und anhaltenden Beschwerden soll eine MRT des  Hüftgelenkes veranlasst werden
Metadata: {}
Score: 0.6730622053146362
Text: . Es ist der SPECT überlegen. Die Autoren ermittelten,  daß der Schmerz der Hauptgrund für die Untersuchung war. Sie schlussfolgerten, daß bei  klinischem Verdacht auf Femurkopfnekrose (anhaltender Leistenschmerz) und normalen  Rön

### Chatbot

In [86]:
class Chatbot:
    def __init__(self, 
                model_type: str = 'openai',
                temperature: float = 0.2,
                max_tokens: int = 100,
                vectordb_path: str = os.path.join(VECTOR_DB_PATH, "medical_dataset_test"),
                collection_name: str = "medical_dataset_test",
                top_p: float = 0.65):
        """
        Initializes the chatbot with medical QA capabilities.

        Args:
            model_type (str): The type of model to use ('openai' or 'huggingface').
        """
        self.model_type = model_type
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.top_p = top_p
        self.vectordb_path = vectordb_path
        self.collection_name = collection_name

        self.llm = self.initialize_model()
        self.retriever = lambda query: retrieve_text(
            vectordb_path=self.vectordb_path,
            query=query,
            collection_name=self.collection_name
        )

    def initialize_model(self):
        """Initialize the LLM with medical-focused parameters"""
        return ChatOpenAI(
            model=GPT_4o,
            temperature=self.temperature,
            max_tokens=self.max_tokens,
            top_p=self.top_p,
            api_key=OPENAI_KEY
        )

    def ask(self, question: str) -> str:
        """
        Ask a question to the chatbot and get an answer.

        Args:
            question (str): The question to ask.

        Returns:
            str: The answer from the chatbot.
        """
        if not question:
            raise ValueError("Question cannot be empty.")

        # Retrieve relevant documents
        matched_texts = self.retriever(question)

        if not matched_texts:
            return "I don't know (no relevant documents found)."
        
        context = "\n".join([doc.page_content for doc, _ in matched_texts])
        
        prompt = ChatPromptTemplate.from_template("""
            Answer this medical question based ONLY on the context below.
            Be concise and factual. If unsure, say 'I don't know'.

            Question: {input}
            Context: {context}

            Answer in the same language as the question:
        """)
        
        formatted_prompt = prompt.format(input=question, context=context)
        
        response = self.llm.invoke(formatted_prompt)
        return response.content

In [87]:
question = "weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?"

bot = Chatbot()
answer = bot.ask(question)
print(answer)

Retrieving from Vector DB Path: backend/app/vector_store/medical_dataset_test
Number of Retrieved Texts: 3
Query used for retrieval: weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?
Ja, weiterführende Bildgebung, wie ein MRT des Hüftgelenkes, ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund.


### Accuracy

In [88]:
evaluation_llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.1, api_key=OPENAI_KEY)
evaluator = load_evaluator("qa", llm=evaluation_llm)

bot = Chatbot()

for item in question_answer_pairs_german:
    answer = bot.ask(item["question"])
    
    eval_result = evaluator.evaluate_strings(
        prediction=answer,
        input=item["question"],
        reference=item["answer_reference"]
    )
    
    print(f"Question: {item['question']}")
    print(f"Answer: {answer}")
    print(f"Reference: {item['answer_reference']}")
    print(f"Evaluation: {eval_result}")  
    print("-" * 80)

Retrieving from Vector DB Path: backend/app/vector_store/medical_dataset_test
Number of Retrieved Texts: 3
Query used for retrieval: Welche weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?
Question: Welche weiterführende Bildgebung ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund?
Answer: Eine MRT des Hüftgelenkes ist indiziert bei persistierenden Hüftschmerzen über 6 Wochen trotz unauffälligem Röntgenbefund.
Reference: Empfehlung: MRT bei persistierenden Beschwerden mit unauffälligem Röntgenbefund.
Evaluation: {'reasoning': 'GRADE: CORRECT', 'value': 'CORRECT', 'score': 1}
--------------------------------------------------------------------------------
Retrieving from Vector DB Path: backend/app/vector_store/medical_dataset_test
Number of Retrieved Texts: 3
Query used for retrieval: Wann sollte eine MRT beider Hüftgelenke durchgeführt werden?
Question: Wann sollte e